In [ ]:
import sys
from pathlib import Path

path_root = Path.cwd().parent

if str(path_root) not in sys.path:
    sys.path.append(str(path_root))

In [ ]:
# test normalizer
import numpy as np
from Normalizer import StdNormalizer

DEVICE = "cuda"
normalizer = StdNormalizer(DEVICE)

mean = 7
scale = 2
e = 0.01  # arbitrary error threshold for random test

data = np.random.normal(mean, scale, 1000000)

normalizer = normalizer.fit(data)
assert normalizer.is_fitted()
print(normalizer.stats.mean - mean)
print(normalizer.stats.scale - scale)
assert abs(normalizer.stats.mean - mean) < e
assert abs(normalizer.stats.scale - scale) < e

data_point = np.random.normal(mean, scale, 1)[0]
transformed_dp = normalizer.transform(data_point).item()
print(f"{data_point} -> {transformed_dp}")
assert (
    transformed_dp > -3 and transformed_dp < 3
)  # standard normal stddev is 1, 99% of the time should be within mean (0) += 3*sd

inverse_transformed_dp = normalizer.inverse_transform(transformed_dp).item()
print(f"{transformed_dp} -> {inverse_transformed_dp}")
print(inverse_transformed_dp - data_point)
assert abs(inverse_transformed_dp - data_point) < e

print("Normalizer tests passed!")

In [ ]:
# test changes in bo/bo
from bayesian_optimization.bo import BO_on_QEC

# bo = BO_on_QEC(code_eval_metric="distance")

# nothing really to test here

In [ ]:
# test changes in bo/objective function
from bayesian_optimization.objective_function import ObjectiveFunction
from code_construction.code_construction import CSSCode
import qldpc

code_class = "bb"

class DummyCodeConstructor:
    def __init__(self):
        self.n = 1

    # def construct(self, code: qldpc.codes.CSSCode):
    #     return CSSCode(code.matrix_x, code.matrix_z)

    def construct(self, matrices):
        return CSSCode(matrices[0], matrices[1])


cc = DummyCodeConstructor()

# NO LONGER WORKS DUE TO TIMEOUT: see test_exact_dist_eval.py
"""
# exact distance
obj = ObjectiveFunction(code_constructor=cc, code_eval_metric="distance", dist_timeout=100)

small_code = qldpc.codes.SteaneCode()
small_code_matrices = np.array([small_code.matrix_x, small_code.matrix_z])
small_known_distance = 3
_, d = obj.forward(small_code_matrices)
assert d == small_known_distance

print("Exact distance test passed!")
print()
"""

# distance estimation with various methods
L = 10
large_code = qldpc.codes.ToricCode(L)
large_code_matrices = np.array([large_code.matrix_x, large_code.matrix_z])

large_known_distance = L
# commented out methods have issues (require license, local binaries missing etc.)
methods = [
    "QDistEvol",
    # 'magmaMinWeight',
    # 'magmaMinWord',
    # 'magmaWEDist',
    # 'qubitSerfBZ',
    # 'qubitSerfMM',
    # 'dist_m4ri_RW',
    # 'dist_m4ri_CC',
    "QDistRndMW",
    "UndetectableErrorStim",
    "GraphLikeErrorStim",
    "ColourCodeDistStim",
    "UndetectableErrorMW",
    "GraphLikeErrorMW",
    "ColourCodeDistMW",
    "decoderDist",
    # 'GurobiDist',
    "MIPDist",
    # 'pySATDist'
]

for m in methods:
    obj = ObjectiveFunction(
        code_constructor=cc, code_eval_metric="distance", dist_method=m
    )

    _, est_d = obj.forward(large_code_matrices)
    print(
        f"method: {m}, actual distance: {large_known_distance}, estimated distance: {est_d}"
    )

print()
print("All methods tested!")